# Mode A: Baseline Pure NSGA-II

**Pure NSGA-II baseline** - No enhancements, no repair heuristics, no RL guidance.

This notebook is the foundation for comparing all other modes (B, C, D, E).

## 1. Imports

In [ ]:
from __future__ import annotations
import random
import numpy as np
from pathlib import Path

from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, run_nsga2, EvolutionConfig, get_best_individual
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary

print("Imports successful")

 All imports from schedule_engine/notebooks/ successful!


## 2. Configuration

In [ ]:
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 50
NGEN = 100
CXPB = 0.9
MUTPB = 0.2

# Fitness weights: -1.0 = minimize both (equal weight)
FITNESS_WEIGHTS = (-1.0, -1.0)

# Evolution config
config = EvolutionConfig(
    pop_size=POP_SIZE,
    ngen=NGEN,
    cxpb=CXPB,
    mutpb=MUTPB,
    fitness_weights=FITNESS_WEIGHTS,
    verbose=True,
    log_interval=10,  # Show detailed breakdown every 10 gens
)

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_a_baseline/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Mode A: pop={POP_SIZE}, ngen={NGEN}, weights={FITNESS_WEIGHTS}")

 Mode A Config: pop=50, ngen=100, cxpb=0.9
 Output: ../output/mode_a_baseline/20260123_093902


## 3. Load Data

In [20]:
# Load all data with single function call
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

[!warn] groups enrolled but courses missing

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (ltp null)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (ltp null)

ENIE 254: BIE4A, BIE4B (ltp null)

ME706: BME7A, BME7B (ltp null)

16 course enrollments skipped

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


## 4. Test Components

In [16]:
# Test individual creation
test_ind = create_random_individual(data)
print(f" Individual has {len(test_ind)} genes")

# Test evaluation
evaluate = create_evaluator(data)
test_fitness = evaluate(test_ind)
print(f" Test fitness: hard={test_fitness[0]}, soft={test_fitness[1]}")

 Individual has 713 genes
 Test fitness: hard=5364.0, soft=2870.0


## 5. Run NSGA-II Evolution

In [17]:
# Run evolution with DRY components
final_pop, stats = run_nsga2(
    data=data,
    config=config,
    create_individual_fn=create_random_individual,
    evaluate_fn=evaluate,
    crossover_fn=course_aware_crossover,
    mutate_fn=lambda ind: smart_mutation(ind, data),  # Closure over data
)

  Gen   0: min_hard=4503, min_soft= 2571, feasible=0/50
  Gen   1: min_hard=3871, min_soft= 2262, feasible=0/50
  Gen   2: min_hard=3520, min_soft= 2128, feasible=0/50
  Gen   3: min_hard=3512, min_soft= 2113, feasible=0/50
  Gen   4: min_hard=3293, min_soft= 1841, feasible=0/50
  Gen   5: min_hard=3039, min_soft= 1669, feasible=0/50
  Gen   6: min_hard=3002, min_soft= 1669, feasible=0/50
  Gen   7: min_hard=2776, min_soft= 1535, feasible=0/50
  Gen   8: min_hard=2642, min_soft= 1473, feasible=0/50
  Gen   9: min_hard=2642, min_soft= 1390, feasible=0/50
  Gen  10: min_hard=2554, min_soft= 1390, feasible=0/50
  Gen  11: min_hard=2359, min_soft= 1390, feasible=0/50
  Gen  12: min_hard=2312, min_soft= 1389, feasible=0/50
  Gen  13: min_hard=2312, min_soft= 1335, feasible=0/50
  Gen  14: min_hard=2268, min_soft= 1327, feasible=0/50
  Gen  15: min_hard=2199, min_soft= 1254, feasible=0/50
  Gen  16: min_hard=2081, min_soft= 1159, feasible=0/50
  Gen  17: min_hard=2064, min_soft= 1159, feasib

## 6. Results & Visualization

In [18]:
# Get best solution
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

# Print summary
print_summary(final_pop, stats, breakdown)

# Plot results
plot_convergence(stats, OUTPUT_DIR / "mode_a_convergence.png", title_prefix="Mode A: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_a_breakdown.png", title="Mode A: Constraint Violations")


 EVOLUTION SUMMARY

 Best Solution:
   Hard Violations: 1552
   Soft Penalty:    656.0
   Feasible:         No

 Final Population (n=50):
   Feasible:     0/50 (0.0%)
   Min Hard:     1552
   Avg Hard:     1622.4
   Min Soft:     557.0
   Avg Soft:     609.1

️ Execution Time: 96.4s

 Best Solution Constraint Breakdown:
    course_completeness: 0
    instructor_exclusivity: 115
    instructor_qualifications: 16
    instructor_schedule_compactness: 89
    instructor_time_availability: 464
    paired_cohort_practical_alignment: 0
    room_exclusivity: 311
    room_suitability: 0
    room_time_availability: 0
    session_continuity: 45
    student_group_exclusivity: 646
    student_lunch_break: 280
    student_schedule_compactness: 242

   Total Hard: 1088, Total Soft: 1120.0

 Saved: ../output/mode_a_baseline/20260123_093902/mode_a_convergence.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 Saved: ../output/mode_a_baseline/20260123_093902/mode_a_breakdown.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Export Results

Generate all outputs: schedule JSON, calendar PDF, plots, and CSVs.

In [ ]:
from schedule_engine.notebooks.export import export_full_results

# Export all results (schedule.json, calendar.pdf, plots, CSVs)
export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_a_baseline",
)

print(f"\n All files saved to: {OUTPUT_DIR}")

 Saved: ../output/mode_a_baseline/20260123_093902/mode_a_baseline_schedule.json
 Saved: ../output/mode_a_baseline/20260123_093902/mode_a_baseline_stats.csv
 Saved: ../output/mode_a_baseline/20260123_093902/mode_a_baseline_summary.json

 All exports complete: ../output/mode_a_baseline/20260123_093902

 All files saved to: ../output/mode_a_baseline/20260123_093902
